# 03 · Cyclone-Event Attribution — Sidr, Aila, Mahasen, Amphan

For each of the four headline cyclones with documented salinity impacts in southwest Bangladesh, we compare the regional-mean NDSI in the 90 days **before** landfall to the 90 days **after**, and bootstrap a 95 % confidence interval on the difference.

**Cyclones evaluated:**

| Event | Landfall | Season | Notes |
|---|---|---|---|
| Sidr | 15 Nov 2007 | Post-monsoon | Cat 4 — Bagerhat, Patuakhali devastation |
| Aila | 25 May 2009 | Pre-monsoon | Multi-year salinity persistence in Satkhira |
| Mahasen | 16 May 2013 | Pre-monsoon | Cat 1 — coastal Bagerhat flooding |
| Amphan | 20 May 2020 | Pre-monsoon | Super cyclone — surge into Satkhira, Khulna |

Hard-coded in `src.config.CYCLONE_EVENTS`; see `docs/cyclone_events.md` for per-storm narrative.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from src.config import DATA_DIR, FIGURES_DIR, RESULTS_DIR, CYCLONE_EVENTS
from src.cyclone_analysis import analyse_all_events
from src.visualization import plot_cyclone_effects

In [ ]:
ts = pd.read_csv(DATA_DIR / 'ndsi_timeseries_2000_2026.csv', parse_dates=['date'])
print(f'Loaded {len(ts)} observations from {ts["date"].min().date()} to {ts["date"].max().date()}')

## 1 · Per-event pre vs post analysis with bootstrap CIs

In [ ]:
results = analyse_all_events(ts)
results

In [ ]:
out_csv = RESULTS_DIR / 'cyclone_event_attribution.csv'
results.to_csv(out_csv, index=False)
print(f'Wrote: {out_csv}')

## 2 · Interpretation

A positive `mean_diff` indicates the post-landfall NDSI was higher than pre-landfall — consistent with surge-driven saline intrusion. The bootstrap CI either including or excluding zero is the basis for an inference call on each storm.

Expected patterns (per `docs/cyclone_events.md`):

- **Sidr (Bagerhat-centred):** moderately positive shift; recovery within 4–6 months.
- **Aila (Satkhira-centred):** large persistent positive shift — the multi-year embankment-failure case.
- **Mahasen (eastern track):** small / near-zero signal in the SW study area — null case.
- **Amphan (Satkhira-centred):** large positive shift; recovery longer than Sidr.

## 3 · Forest-plot summary figure

In [ ]:
fig_path = FIGURES_DIR / 'fig_cyclone_impacts.png'
plot_cyclone_effects(results, save_to=fig_path)
print(f'Saved: {fig_path}')

---

The cyclone-recency signal extracted here feeds the Random Forest in notebook **05** as the `years_since_cyclone` feature. The pre/post mean shifts also inform the projection assumption that cyclone landfalls continue at their historical cadence.